# Data Pipeline — Amazon Reviews 2023 (Appliances)
Exploratory analysis on a streamed sample, followed by schema design and ingestion.

> **Note:** `datasets` >= 3.x removed support for dataset loading scripts
> (`RuntimeError: Dataset scripts are no longer supported`). The raw `.jsonl` file is
> streamed directly via the `json` builder with an `hf://` path instead.

In [4]:
from datasets import load_dataset

# datasets >= 3.x no longer runs dataset scripts; stream the raw jsonl file directly
reviews_stream = load_dataset(
    "json",
    data_files="hf://datasets/McAuley-Lab/Amazon-Reviews-2023/raw/review_categories/Appliances.jsonl",
    split="train",
    streaming=True,
)

sample = []
for i, record in enumerate(reviews_stream):
    if i >= 5000:
        break
    sample.append(record)

print(f"Collected {len(sample)} records")
print(sample[0])

Collected 5000 records
{'rating': 5.0, 'title': 'Work great', 'text': 'work great. use a new one every month', 'images': [], 'asin': 'B01N0TQ0OH', 'parent_asin': 'B01N0TQ0OH', 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ', 'timestamp': 1519317108692, 'helpful_vote': 0, 'verified_purchase': True}


In [5]:
import pandas as pd
df = pd.DataFrame(sample)

print("Shape:", df.shape)
print("\n--- Null counts ---")
print(df.isnull().sum())
print("\n--- Rating distribution ---")
print(df["rating"].value_counts().sort_index())
print("\n--- Timestamp range ---")
ts = pd.to_datetime(df["timestamp"], unit="ms")
print("min:", ts.min(), "| max:", ts.max())
print("\n--- Text length (chars) ---")
print(df["text"].str.len().describe().round(1))

Shape: (5000, 10)

--- Null counts ---
rating               0
title                0
text                 0
images               0
asin                 0
parent_asin          0
user_id              0
timestamp            0
helpful_vote         0
verified_purchase    0
dtype: int64

--- Rating distribution ---
rating
1.0     376
2.0     170
3.0     293
4.0     582
5.0    3579
Name: count, dtype: int64

--- Timestamp range ---
min: 2007-03-29 23:05:20 | max: 2023-03-17 07:53:04.630000

--- Text length (chars) ---
count     5000.0
mean       248.8
std        454.4
min          1.0
25%         47.0
50%        116.0
75%        279.0
max      11328.0
Name: text, dtype: float64


## Sample EDA — findings

- **Nulls:** none in the 5k sample; ingest still handles nulls defensively (sample ≠ full set).
- **Rating distribution:** J-shaped (72% five-star, 7.5% one-star) — typical for Amazon reviews.
  Averages alone are misleading; distribution will be reported alongside.
- **Timestamp:** 13-digit epoch **milliseconds** (verified: 2007–2023 range after `unit="ms"`).
  Will be converted to seconds (`// 1000`) at ingestion.
- **Text length:** heavily right-skewed (median 116 chars, max 11,328). Most reviews fit a
  single chunk; a long tail needs chunking. Reviews under ~20 chars will be excluded from
  the retrieval corpus.

In [6]:
import sqlite3
import pandas as pd

df = pd.DataFrame(sample)

# Convert ms -> s. Int64 (nullable) looks like the right choice:
# it can safely hold NaN if any timestamp is missing.
ts_seconds = (pd.to_numeric(df["timestamp"], errors="coerce") // 1000).astype("Int64")

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE reviews_test (ts INTEGER)")
conn.executemany(
    "INSERT INTO reviews_test (ts) VALUES (?)",
    [(v,) for v in ts_seconds],
)

# Diagnosis: what type did SQLite ACTUALLY store?
print(conn.execute(
    "SELECT typeof(ts), COUNT(*) FROM reviews_test GROUP BY typeof(ts)"
).fetchall())

[('blob', 5000)]


In [7]:
import numpy as np

# Link 1: what does the Int64 (nullable) column yield per element?
v = ts_seconds.iloc[0]
print("element type:", type(v))

# Link 2: does Python's sqlite3 recognize it as an int?
print("isinstance int:", isinstance(v, int))

# Link 3: minimal reproduction — numpy.int64 straight into SQLite
c2 = sqlite3.connect(":memory:")
c2.execute("CREATE TABLE t (v INTEGER)")
c2.execute("INSERT INTO t VALUES (?)", (np.int64(5),))
print("np.int64 stored as:", c2.execute("SELECT typeof(v) FROM t").fetchone())

# Contrast: a plain Python int
c2.execute("INSERT INTO t VALUES (?)", (int(5),))
print("all types now:", c2.execute("SELECT typeof(v), COUNT(*) FROM t GROUP BY typeof(v)").fetchall())

element type: <class 'numpy.int64'>
isinstance int: False
np.int64 stored as: ('blob',)
all types now: [('blob', 1), ('integer', 1)]


In [8]:
# Fix: convert each value to a native Python int (None-safe)
ts_fixed = [int(v) if pd.notna(v) else None for v in ts_seconds]

c3 = sqlite3.connect(":memory:")
c3.execute("CREATE TABLE reviews_test (ts INTEGER)")
c3.executemany(
    "INSERT INTO reviews_test (ts) VALUES (?)",
    [(v,) for v in ts_fixed],
)

print(c3.execute(
    "SELECT typeof(ts), COUNT(*) FROM reviews_test GROUP BY typeof(ts)"
).fetchall())

[('integer', 5000)]


## Type-integrity finding: pandas nullable Int64 → SQLite BLOB

**Symptom:** `ts` values stored as BLOB despite the column being declared INTEGER —
silently, with no error. Integer operations (`MIN`, `MAX`, comparisons) break downstream.

**Root cause:** pandas' nullable `Int64` dtype yields `numpy.int64` scalars, which fail
`isinstance(v, int)` — so `sqlite3` doesn't recognize them as integers. Since numpy scalars
support the buffer protocol, sqlite3 falls back to storing their raw 8 bytes as BLOB.
SQLite's type affinity is a suggestion, not a constraint, so the INTEGER declaration
doesn't prevent this.

**Fix:** coerce each value to a native Python `int` before insertion
(`int(v) if pd.notna(v) else None`). Verified: `typeof(ts)` → `integer` for all rows.
The ingestion module will include a regression test for this.

In [9]:
meta_stream = load_dataset(
    "json",
    data_files="hf://datasets/McAuley-Lab/Amazon-Reviews-2023/raw/meta_categories/meta_Appliances.jsonl",
    split="train",
    streaming=True,
)

meta_sample = []
for i, record in enumerate(meta_stream):
    if i >= 1000:
        break
    meta_sample.append(record)

meta_df = pd.DataFrame(meta_sample)
print("Shape:", meta_df.shape)
print("\nColumns:", list(meta_df.columns))
print("\n--- Null counts (top) ---")
print(meta_df.isnull().sum().sort_values(ascending=False).head(10))
print("\n--- rating_number distribution ---")
print(meta_df["rating_number"].describe().round(1))

Shape: (1000, 14)

Columns: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']

--- Null counts (top) ---
bought_together    1000
price               435
store                11
main_category         4
average_rating        0
title                 0
description           0
features              0
rating_number         0
images                0
dtype: int64

--- rating_number distribution ---
count     1000.0
mean       228.7
std       1285.8
min          1.0
25%          5.0
50%         22.0
75%         93.2
max      30959.0
Name: rating_number, dtype: float64


## Schema

Two tables joined on `parent_asin`. Notable choices: `price` is nullable (43.5% missing
in the sample), `ts` is INTEGER epoch **seconds**, `verified` is INTEGER 0/1 (SQLite has
no native boolean). `IF NOT EXISTS` keeps initialization idempotent.

In [11]:
SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS products (
    parent_asin    TEXT PRIMARY KEY,
    title          TEXT NOT NULL,
    store          TEXT,
    main_category  TEXT,
    price          REAL,
    average_rating REAL,
    rating_number  INTEGER,
    details_json   TEXT
);
CREATE TABLE IF NOT EXISTS reviews (
    review_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    parent_asin    TEXT REFERENCES products(parent_asin),
    rating         REAL NOT NULL,
    title          TEXT,
    text           TEXT,
    helpful_vote   INTEGER,
    verified       INTEGER,
    ts             INTEGER
);
CREATE INDEX IF NOT EXISTS idx_reviews_asin ON reviews(parent_asin);
CREATE INDEX IF NOT EXISTS idx_reviews_ts   ON reviews(ts);
"""

def init_db(conn: sqlite3.Connection) -> None:
    conn.executescript(SCHEMA_SQL)
    conn.commit()

# quick check
test_conn = sqlite3.connect(":memory:")
init_db(test_conn)
tables = test_conn.execute(
    "SELECT name FROM sqlite_master WHERE type IN ('table','index') ORDER BY name"
).fetchall()
print(tables)

[('idx_reviews_asin',), ('idx_reviews_ts',), ('products',), ('reviews',), ('sqlite_autoindex_products_1',), ('sqlite_sequence',)]


## Normalization functions

Raw JSON records → typed rows. Defensive `.get()` access throughout (nulls expected in
the full set). `normalize_reviews` applies the type-integrity lesson per value:
timestamps become native Python `int` seconds, booleans become 0/1, `None` passes through.
Records without a title are skipped.

In [16]:
import json

def normalize_products(records: list[dict]) -> list[tuple]:
    """Meta records -> rows for the products table. Skips records without title."""
    rows = []
    for r in records:
        title = r.get("title")
        if not title:
            continue
        desc = r.get("description") or []
        features = r.get("features") or []
        rows.append((
            r.get("parent_asin"),
            title,
            r.get("store"),
            r.get("main_category"),
            r.get("price"),
            r.get("average_rating"),
            r.get("rating_number"),
            json.dumps({"description": desc, "features": features}),
        ))
    return rows


def normalize_reviews(records: list[dict]) -> list[tuple]:
    """Review records -> rows for the reviews table."""
    rows = []
    for r in records:
        ts_ms = r.get("timestamp")

        # ms -> s, native Python int; None stays None (BLOB lesson applied per-value)
        ts = int(ts_ms // 1000) if ts_ms is not None else None

        # bool -> 1/0; None stays None
        vp = r.get("verified_purchase")
        verified = int(vp) if vp is not None else None

        rows.append((
            r.get("parent_asin"),
            r.get("rating"),
            r.get("title"),
            r.get("text"),
            r.get("helpful_vote"),
            verified,
            ts,
        ))
    return rows

p_rows = normalize_products(meta_sample)
r_rows = normalize_reviews(sample)

print(f"products: {len(p_rows)} rows (from {len(meta_sample)})")
print(f"reviews:  {len(r_rows)} rows (from {len(sample)})")
print("first review row:", r_rows[0])
print("ts type:", type(r_rows[0][6]))

products: 999 rows (from 1000)
reviews:  5000 rows (from 5000)
first review row: ('B01N0TQ0OH', 5.0, 'Work great', 'work great. use a new one every month', 0, 1, 1519317108)
ts type: <class 'int'>


## End-to-end smoke test

Writing both samples into an in-memory database: row counts, `typeof(ts)` verification,
and a join proving the two tables connect via `parent_asin`. Note: the review and meta
samples were drawn independently (first 5k reviews vs. first 1k products), so join
overlap may be small — the full ingestion in the next section aligns them.

In [17]:
conn = sqlite3.connect(":memory:")
init_db(conn)

conn.executemany(
    "INSERT OR IGNORE INTO products VALUES (?,?,?,?,?,?,?,?)", p_rows
)
conn.executemany(
    "INSERT INTO reviews (parent_asin, rating, title, text, helpful_vote, verified, ts) "
    "VALUES (?,?,?,?,?,?,?)", r_rows
)
conn.commit()

# sanity: row counts + ts really stored as integer + a join works
print(conn.execute("SELECT COUNT(*) FROM products").fetchone())
print(conn.execute("SELECT COUNT(*) FROM reviews").fetchone())
print(conn.execute("SELECT typeof(ts) FROM reviews LIMIT 1").fetchone())
print(conn.execute("""
    SELECT p.title, COUNT(r.review_id) AS n
    FROM products p JOIN reviews r ON r.parent_asin = p.parent_asin
    GROUP BY p.parent_asin ORDER BY n DESC LIMIT 3
""").fetchall())

(999,)
(5000,)
('integer',)
[('Cuisinart Replacement Water Filters, 2-Pack', 14), ('Pro Mael Disposable Coffee Filters Paper for Keurig Brewers Single Serve 1.0 and 2.0 Use with All Brands Reusable K Cup Pods, White 720 Filters (Pack of 2)', 11), ('PARTY BARGAINS 600 Paper Coffee Filters - White Classic Design Single-Use Coffee Filter Compatible for Keurig 1.0 & 2.0, Perfect Size and Quantity', 9)]


## Scaling to the full set

The pipeline above was developed and validated on streamed samples. The same
normalization logic, applied to the full Appliances category with the sizing filter
(`300 <= rating_number < 1000`), produces `data/products.db`. The cells below verify
the full database against expectations.

In [18]:
full = sqlite3.connect("file:../data/products.db?mode=ro", uri=True)

print("products:", full.execute("SELECT COUNT(*) FROM products").fetchone())
print("reviews: ", full.execute("SELECT COUNT(*) FROM reviews").fetchone())
print("ts types:", full.execute(
    "SELECT typeof(ts), COUNT(*) FROM reviews GROUP BY typeof(ts)"
).fetchall())
print("band check:", full.execute(
    "SELECT MIN(rating_number), MAX(rating_number) FROM products"
).fetchone())
print("date range:", full.execute(
    "SELECT datetime(MIN(ts),'unixepoch'), datetime(MAX(ts),'unixepoch') FROM reviews"
).fetchone())

products: (5006,)
reviews:  (441857,)
ts types: [('integer', 441857)]
band check: (300, 999)
date range: ('2003-12-02 23:33:28', '2023-09-09 19:24:12')
